# Transfer Learning using ResNet50

## Transfer Learning

Transfer learning means using a model that was already trained on a large dataset and adapting it to a new task. For CIFAR10, we use ResNet50 pretrained on ImageNet, then add new layers to classify our 10 target classes.

Benefits of transfer learning:

* saves training time
* works well with smaller datasets
* reuses useful visual features like edges and textures

## ResNet50

ResNet50 (Residual Network-50) is one of the most influential deep convolutional neural networks in computer vision. 

**Historical Background**

Before ResNet:

* CNNs were becoming deeper and deeper.
* Very deep networks suffered from the **vanishing gradient problem**.
* Adding more layers often resulted in worse training performance.

ResNet introduced a revolutionary idea:

> Instead of learning a direct mapping, learn the residual (difference) between input and output.

This allowed networks to become much deeper while remaining trainable.

Its design philosophy was:

* residual learning
* extremely deep architectures
* easier optimization

---

### Why ResNet Was Revolutionary

ResNet introduced several key ideas:

| Innovation           | Importance                |
| -------------------- | ------------------------- |
| Residual Connections | Solve vanishing gradients |
| Deep Architectures   | Train very deep networks  |
| Identity Mapping     | Easier optimization       |
| Bottleneck Blocks    | Computational efficiency  |

The key innovation is the **Skip Connection**:

$$
y = F(x) + x
$$

where:

* $x$ = input
* $F(x)$ = learned residual mapping
* $y$ = output

Instead of learning:

$$
H(x)
$$

ResNet learns:

$$
F(x)=H(x)-x
$$

which is easier to optimize.

---

### Main Idea of ResNet50

Traditional CNNs stack layers sequentially:

$$
x \rightarrow Layer_1 \rightarrow Layer_2 \rightarrow Output
$$

ResNet introduces shortcut paths:

$$
x \rightarrow F(x)
$$

and then

$$
Output = F(x) + x
$$

This enables gradients to flow directly through the network during backpropagation.

---

![ResNet Architecture](https://cdn.educba.com/academy/wp-content/uploads/2022/10/Keras-ResNet50.jpg)

Image source: https://cdn.educba.com/academy/wp-content/uploads/2022/10/Keras-ResNet50.jpg

---

### Input Layer (Original ResNet50 on ImageNet)

Standard ResNet50 uses:

$$
224 \times 224 \times 3
$$

images.

The original architecture begins with:

| Layer | Details                        |
| ----- | ------------------------------ |
| Conv1 | 7×7 Conv, 64 filters, stride 2 |
| Pool1 | 3×3 MaxPool, stride 2          |

---

## CIFAR10 Adaptation

**Important:** CIFAR10 images are much smaller than ImageNet images.

| Property       | ImageNet | CIFAR10 | Our Setup |
| -------------- | -------- | ------- | --------- |
| Original Size  | 224×224  | 32×32   | 32×32     |
| After Upsample | -        | -       | 64×64     |
| Channels       | 3        | 3       | 3         |

**Why Upsample?**

* ResNet50 was pretrained on ImageNet.
* CIFAR10 images are only 32×32.
* Upsampling to 64×64 provides larger feature maps.
* Improves transfer learning performance while keeping computation manageable.

## Importing libraries

In [1]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input

## Load and prepare data

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

 40828928/170498071 ━━━━━━━━━━━━━━━━━━━━ 44:16 20us/step

In [4]:
x_train = preprocess_input(x_train.astype('float32'))
x_test = preprocess_input(x_test.astype('float32'))

## One-Hot Encoding

In [5]:
num_classes = len(np.unique(y_train))

y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

## Load ResNet50 Base Model

In [6]:
# Load pretrained ResNet50 model with ImageNet weights
resnet_base = ResNet50(
    weights='imagenet',              # Use weights trained on ImageNet (1000 classes)
    include_top=False,               # Remove classification head - we'll add custom layers for CIFAR10
    input_shape=(64, 64, 3)          # Input: 64×64×3 (upsampled from CIFAR10's 32×32)
)

# Freeze base model weights - don't update during training
# Keep pretrained features from ImageNet, only train new custom layers
resnet_base.trainable = False        # Prevents weight updates in ResNet50 layers

In [7]:
resnet_base.summary()

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 70, 70, 3) │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 32, 32,    │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 32, 32,    │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 32, 32,    │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 34, 34,    │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 16, 16,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 16, 16,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 16, 16,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 16, 16,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 16, 16,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 16, 16,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 16, 16,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 16, 16,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 16, 16,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 16, 16,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 16, 16,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

## Prepare the model

In [8]:
# Build transfer learning model for CIFAR10
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),           # Input: original CIFAR10 image (32×32×3)

    layers.UpSampling2D(size=(2, 2)),          # Upsample to 64×64 (required by ResNet50)

    resnet_base,                               # Pretrained ResNet50 feature extractor
    layers.Flatten(),                          # Flatten 2D feature maps to 1D vector

    # Custom dense layers for CIFAR10 classification
    layers.Dense(128, activation='relu'),      # Feature refinement layer
    layers.Dense(64, activation='relu'),       # Further dimensionality reduction

    layers.Dense(10, activation='softmax'),    # Output: 10 classes (softmax for multiclass)
])

In [9]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ up_sampling2d (UpSampling2D)    │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 2, 2, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     1,048,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,645,322 (94.01 MB)

 Trainable params: 1,057,610 (4.03 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

## Compile the model

In [10]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## Train the model

In [ ]:
history = model.fit(
    x_train,
    y_train,
    batch_size=64,           # Process 64 samples at a time before updating weights
    epochs=3,                # Train for 3 complete passes through the dataset
    validation_split=0.1,    # Use 10% of training data (5000 samples) to validate during training
)
# Returns training history with loss and accuracy metrics for each epoch

Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 1910s 3s/step - accuracy: 0.7608 - loss: 0.7184 - val_accuracy: 0.8028 - val_loss: 0.5995
Epoch 2/3
166/704 ━━━━━━━━━━━━━━━━━━━━ 21:54 2s/step - accuracy: 0.8325 - loss: 0.4787

## Evaluate the model

In [ ]:
# Evaluate model performance on test set
# Returns [loss, accuracy]
model.evaluate(x_test, y_test)   # Measure final performance on unseen test data

313/313 ━━━━━━━━━━━━━━━━━━━━ 141s 449ms/step - accuracy: 0.7969 - loss: 0.6906


[0.6906166076660156, 0.7968999743461609]

In [ ]:
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

## Visualization

In [ ]:
import random
idx = random.randint(0, len(x_test) - 1)
pred = model.predict(x_test[idx:idx+1])
predicted_class = np.argmax(pred)
actual_class = np.argmax(y_test[idx])
class_names[predicted_class]

plt.imshow(x_test[idx])
plt.title(f'pred: {class_names[predicted_class]} | actual: {class_names[actual_class]}')
plt.axis('off')
plt.show()

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Tools for computing and plotting confusion matrix

y_pred = model.predict(x_test)  # Model predictions for all test images

y_pred_classes = np.argmax(y_pred, axis=1)  # Convert predicted probabilities to class labels
y_true = np.argmax(y_test, axis=1)          # Convert one-hot test labels back to class indices

cm = confusion_matrix(y_true, y_pred_classes)  # Compute confusion matrix
# Create a visual display of the confusion matrix with class labels
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)

fig, ax = plt.subplots(figsize=(10, 10))  # Plot size for readability
disp.plot(ax=ax, cmap=plt.cm.Blues)
plt.xticks(rotation=45)  # Rotate x labels so class names are easier to read
plt.show()

## Saving the model

In [ ]:
model.save('outputs/resnet50_model.keras')